# LAB2 - Bài 2 (Trang 32): F-measure của các bài tập cơ bản

Đinh Nhật Huyền Nhân - 6451071055 - CQ.64.CNTT

Tự cài đặt ID3, CART (chia nhiều nhánh như trong tài liệu) và Naïve Bayes làm trơn Laplace, sau đó tính precision, recall và F-measure trên các hồ sơ kiểm thử có nhãn thực tế trong đề.

- Khi các thuộc tính hòa điểm, chọn thuộc tính đứng trước trong bảng dữ liệu.
- Khi một phép chia không xác định (không có dự đoán dương, hoặc P + R = 0), giá trị được tính là 0.

### Cài đặt các thuật toán

In [1]:
from collections import Counter
from math import log2


def entropy(labels):
    n = len(labels)
    return -sum(c / n * log2(c / n) for c in Counter(labels).values())


def gini(labels):
    n = len(labels)
    return 1 - sum((c / n) ** 2 for c in Counter(labels).values())


def split_scores(rows, attrs, target, criterion):
    """Trả về điểm của từng thuộc tính: độ lợi thông tin (ID3) hoặc Gini_A (CART)."""
    labels = [r[target] for r in rows]
    scores = {}
    for a in attrs:
        groups = {}
        for r in rows:
            groups.setdefault(r[a], []).append(r[target])
        if criterion == 'id3':
            scores[a] = entropy(labels) - sum(len(g) / len(rows) * entropy(g) for g in groups.values())
        else:
            scores[a] = sum(len(g) / len(rows) * gini(g) for g in groups.values())
    return scores


def build_tree(rows, attrs, target, criterion, domains):
    labels = [r[target] for r in rows]
    majority = Counter(labels).most_common(1)[0][0]
    if len(set(labels)) == 1 or not attrs:
        return {'leaf': majority, 'n': len(rows)}
    scores = split_scores(rows, attrs, target, criterion)
    # Khi hòa điểm: chọn thuộc tính đứng trước trong bảng dữ liệu
    # (làm tròn để sai số dấu phẩy động không làm mất trường hợp hòa)
    rounded = {a: round(s, 10) for a, s in scores.items()}
    pick = max if criterion == 'id3' else min
    best = pick(attrs, key=lambda a: rounded[a])
    node = {'attr': best, 'majority': majority, 'scores': scores, 'children': {}}
    rest = [a for a in attrs if a != best]
    for v in domains[best]:
        subset = [r for r in rows if r[best] == v]
        if subset:
            node['children'][v] = build_tree(subset, rest, target, criterion, domains)
    return node


def predict_tree(node, x):
    while 'leaf' not in node:
        child = node['children'].get(x[node['attr']])
        if child is None:  # Nhánh không có mẫu huấn luyện: lấy lớp đa số của nút cha
            return node['majority']
        node = child
    return node['leaf']


def print_tree(node, indent=''):
    if 'leaf' in node:
        print(f"{indent}=> {node['leaf']} ({node['n']} mẫu)")
        return
    for v, child in node['children'].items():
        print(f"{indent}[{node['attr']} = {v}]")
        print_tree(child, indent + '    ')


def naive_bayes(rows, attrs, target, domains, x):
    """Naïve Bayes làm trơn Laplace, trả về lớp dự đoán và điểm của từng lớp."""
    classes = domains[target]
    scores = {}
    for c in classes:
        rows_c = [r for r in rows if r[target] == c]
        p = (len(rows_c) + 1) / (len(rows) + len(classes))
        for a in attrs:
            count = sum(r[a] == x[a] for r in rows_c)
            p *= (count + 1) / (len(rows_c) + len(domains[a]))
        scores[c] = p
    return max(classes, key=lambda c: scores[c]), scores


def conflicts(rows, attrs, target):
    groups = {}
    for i, r in enumerate(rows, 1):
        groups.setdefault(tuple(r[a] for a in attrs), []).append((i, r[target]))
    return [g for g in groups.values() if len({c for _, c in g}) > 1]


def prf(actual, pred, positive):
    tp = sum(a == positive and p == positive for a, p in zip(actual, pred))
    fp = sum(a != positive and p == positive for a, p in zip(actual, pred))
    fn = sum(a == positive and p != positive for a, p in zip(actual, pred))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f

### Dữ liệu từ tài liệu thực hành

In [2]:
def table(cols, text):
    return [dict(zip(cols, [v.strip() for v in line.split('|')])) for line in text.strip().splitlines()]


credit_cols = ['Thu nhập', 'Nghề nghiệp', 'Tình trạng hôn nhân', 'Sở hữu nhà', 'Nguy cơ']
credit = table(credit_cols, """
Thấp|Vận tải|Độc thân|Rồi|Rủi ro cao
Thấp|Vận tải|Độc thân|Chưa|Rủi ro cao
Cao|Vận tải|Độc thân|Rồi|Rủi ro thấp
Trung bình|Truyền thông|Độc thân|Rồi|Rủi ro thấp
Trung bình|Kinh doanh|Đã kết hôn|Rồi|Rủi ro thấp
Trung bình|Kinh doanh|Đã kết hôn|Chưa|Rủi ro cao
Cao|Kinh doanh|Đã kết hôn|Chưa|Rủi ro thấp
Thấp|Truyền thông|Độc thân|Rồi|Rủi ro cao
Thấp|Kinh doanh|Đã kết hôn|Rồi|Rủi ro thấp
Trung bình|Truyền thông|Đã kết hôn|Rồi|Rủi ro thấp
Thấp|Truyền thông|Đã kết hôn|Chưa|Rủi ro thấp
Cao|Truyền thông|Độc thân|Chưa|Rủi ro thấp
Cao|Vận tải|Đã kết hôn|Rồi|Rủi ro thấp
Trung bình|Truyền thông|Độc thân|Chưa|Rủi ro cao
Cao|Kinh doanh|Độc thân|Rồi|Rủi ro cao
""")
credit_test = table(credit_cols, """
Trung bình|Vận tải|Độc thân|Rồi|Rủi ro cao
Cao|Truyền thông|Độc thân|Rồi|Rủi ro thấp
Thấp|Vận tải|Đã kết hôn|Rồi|Rủi ro thấp
""")

toy_cols = ['Loại', 'Số màu', 'Kích thước', 'Chất liệu', 'Doanh số bán']
toy = table(toy_cols, """
Điều khiển|3|Nhỏ|Nhựa PP|Cao
Xếp hình|5|Vừa|Cao su|Thấp
Xếp hình|7|To|Nhựa PP|Thấp
Điều khiển|5|Nhỏ|Cao su|Thấp
Búp bê|3|Vừa|Nhựa PP|Thấp
Điều khiển|5|Vừa|Nhựa PP|Cao
Búp bê|5|To|Nhựa PP|Cao
Điều khiển|7|Vừa|Cao su|Thấp
Xếp hình|7|To|Cao su|Cao
Xếp hình|3|To|Nhựa PP|Thấp
Búp bê|3|Nhỏ|Cao su|Thấp
Xếp hình|3|Nhỏ|Nhựa PP|Cao
Điều khiển|5|To|Cao su|Thấp
Búp bê|5|Vừa|Nhựa PP|Cao
Búp bê|7|To|Nhựa PP|Cao
""")
toy_test = table(toy_cols, """
Búp bê|3|To|Cao su|Thấp
Xếp hình|5|To|Nhựa PP|Thấp
Điều khiển|3|Vừa|Cao su|Cao
""")

senti_cols = ['giảm', 'người', 'chuyển', 'yêu', 'vừa', 'đi', 'Cảm xúc']
senti = table(senti_cols, """
0..5|11..20|>20|11..20|>20|0..5|tốt
11..20|6..10|6..10|0..5|11..20|11..20|tốt
6..10|0..5|6..10|11..20|0..5|6..10|xấu
>20|0..5|11..20|6..10|0..5|>20|bình thường
0..5|>20|11..20|0..5|6..10|0..5|xấu
0..5|6..10|0..5|0..5|11..20|11..20|xấu
0..5|6..10|11..20|0..5|6..10|0..5|tốt
11..20|>20|0..5|11..20|0..5|11..20|bình thường
0..5|0..5|6..10|6..10|6..10|>20|tốt
11..20|0..5|11..20|11..20|0..5|11..20|tốt
>20|6..10|0..5|0..5|0..5|6..10|xấu
0..5|0..5|11..20|0..5|11..20|>20|bình thường
6..10|11..20|6..10|>20|0..5|6..10|bình thường
11..20|6..10|>20|11..20|0..5|0..5|xấu
""")
senti_test = table(senti_cols, """
0..5|6..10|0..5|11..20|6..10|0..5|xấu
0..5|0..5|6..10|0..5|11..20|>20|tốt
6..10|0..5|11..20|>20|6..10|6..10|bình thường
6..10|11..20|6..10|6..10|>20|0..5|tốt
""")

### Hàm đánh giá

In [3]:
print("Đinh Nhật Huyền Nhân - 6451071055")
import pandas as pd

DATASETS = {
    'III.1 Rủi ro tín dụng': (credit, credit_test, credit_cols),
    'IV.1 Doanh số đồ chơi': (toy, toy_test, toy_cols),
    'IV.2 Phân tích cảm xúc': (senti, senti_test, senti_cols),
}
summary = []


def evaluate(name):
    rows, test, cols = DATASETS[name]
    attrs, target = cols[:-1], cols[-1]
    domains = {c: sorted({r[c] for r in rows + test}) for c in cols}
    print('Mâu thuẫn trong dữ liệu:', conflicts(rows, attrs, target) or 'không có')

    print('\nĐộ lợi thông tin tại nút gốc:')
    for a, s in split_scores(rows, attrs, target, 'id3').items():
        print(f'  {a}: {s:.3f}')
    print('Chỉ số Gini tại nút gốc:')
    for a, s in split_scores(rows, attrs, target, 'cart').items():
        print(f'  {a}: {s:.3f}')

    id3_tree = build_tree(rows, attrs, target, 'id3', domains)
    cart_tree = build_tree(rows, attrs, target, 'cart', domains)
    print('\nCây ID3:')
    print_tree(id3_tree)
    print('\nCây CART:')
    print_tree(cart_tree)

    actual = [r[target] for r in test]
    predictions = {
        'ID3': [predict_tree(id3_tree, r) for r in test],
        'CART': [predict_tree(cart_tree, r) for r in test],
        'Naive Bayes': [naive_bayes(rows, attrs, target, domains, r)[0] for r in test],
    }
    print('\nThực tế:', actual)
    for method, pred in predictions.items():
        print(f'{method}:', pred)

    print()
    for method, pred in predictions.items():
        scores = [prf(actual, pred, c) for c in domains[target]]
        for c, (p, r, f) in zip(domains[target], scores):
            summary.append({'Bài': name, 'Phương pháp': method, 'Lớp': c,
                            'Precision': p, 'Recall': r, 'F-measure': f})
        summary.append({'Bài': name, 'Phương pháp': method, 'Lớp': 'Trung bình macro',
                        'Precision': sum(s[0] for s in scores) / len(scores),
                        'Recall': sum(s[1] for s in scores) / len(scores),
                        'F-measure': sum(s[2] for s in scores) / len(scores)})
    table_ = pd.DataFrame([s for s in summary if s['Bài'] == name]).drop(columns='Bài')
    print(table_.round(3).to_string(index=False))

Đinh Nhật Huyền Nhân - 6451071055


### III.1 Rủi ro tín dụng (dùng để đối chiếu với kết quả trong tài liệu)

In [4]:
print("Đinh Nhật Huyền Nhân - 6451071055")
evaluate('III.1 Rủi ro tín dụng')

Đinh Nhật Huyền Nhân - 6451071055
Mâu thuẫn trong dữ liệu: không có

Độ lợi thông tin tại nút gốc:
  Thu nhập: 0.083
  Nghề nghiệp: 0.013
  Tình trạng hôn nhân: 0.186
  Sở hữu nhà: 0.020
Chỉ số Gini tại nút gốc:
  Thu nhập: 0.427
  Nghề nghiệp: 0.471
  Tình trạng hôn nhân: 0.364
  Sở hữu nhà: 0.467

Cây ID3:
[Tình trạng hôn nhân = Đã kết hôn]
    [Thu nhập = Cao]
        => Rủi ro thấp (2 mẫu)
    [Thu nhập = Thấp]
        => Rủi ro thấp (2 mẫu)
    [Thu nhập = Trung bình]
        [Sở hữu nhà = Chưa]
            => Rủi ro cao (1 mẫu)
        [Sở hữu nhà = Rồi]
            => Rủi ro thấp (2 mẫu)
[Tình trạng hôn nhân = Độc thân]
    [Thu nhập = Cao]
        [Nghề nghiệp = Kinh doanh]
            => Rủi ro cao (1 mẫu)
        [Nghề nghiệp = Truyền thông]
            => Rủi ro thấp (1 mẫu)
        [Nghề nghiệp = Vận tải]
            => Rủi ro thấp (1 mẫu)
    [Thu nhập = Thấp]
        => Rủi ro cao (3 mẫu)
    [Thu nhập = Trung bình]
        [Sở hữu nhà = Chưa]
            => Rủi ro cao (1

**Nhận xét:** Với lớp dương là Rủi ro thấp như trong tài liệu, cây có P = 2/3 ≈ 0,667, R = 1 nên F = 2 × 0,667 × 1 / (0,667 + 1) = 0,8. Naïve Bayes có P = R = F = 1. Độ lợi thông tin, độ chính xác và độ phủ đều khớp tài liệu. Riêng Gini của Tình trạng hôn nhân khi tính đúng là 8/15 × 0,469 + 7/15 × 0,245 ≈ 0,364 (tài liệu ghi 0,403), nhưng thuộc tính được chọn không đổi. Tại nút Đã kết hôn có hai thuộc tính hòa điểm: bài chọn Thu nhập, còn phần CART của tài liệu chọn Sở hữu nhà; dự đoán cho 3 hồ sơ vẫn như nhau.

### IV.1 Doanh số đồ chơi

In [5]:
print("Đinh Nhật Huyền Nhân - 6451071055")
evaluate('IV.1 Doanh số đồ chơi')

Đinh Nhật Huyền Nhân - 6451071055
Mâu thuẫn trong dữ liệu: không có

Độ lợi thông tin tại nút gốc:
  Loại: 0.026
  Số màu: 0.006
  Kích thước: 0.006
  Chất liệu: 0.186
Chỉ số Gini tại nút gốc:
  Loại: 0.480
  Số màu: 0.493
  Kích thước: 0.493
  Chất liệu: 0.378

Cây ID3:
[Chất liệu = Cao su]
    [Loại = Búp bê]
        => Thấp (1 mẫu)
    [Loại = Xếp hình]
        [Số màu = 5]
            => Thấp (1 mẫu)
        [Số màu = 7]
            => Cao (1 mẫu)
    [Loại = Điều khiển]
        => Thấp (3 mẫu)
[Chất liệu = Nhựa PP]
    [Loại = Búp bê]
        [Số màu = 3]
            => Thấp (1 mẫu)
        [Số màu = 5]
            => Cao (2 mẫu)
        [Số màu = 7]
            => Cao (1 mẫu)
    [Loại = Xếp hình]
        [Kích thước = Nhỏ]
            => Cao (1 mẫu)
        [Kích thước = To]
            => Thấp (2 mẫu)
    [Loại = Điều khiển]
        => Cao (2 mẫu)

Cây CART:
[Chất liệu = Cao su]
    [Loại = Búp bê]
        => Thấp (1 mẫu)
    [Loại = Xếp hình]
        [Số màu = 5]
            =

**Nhận xét:** Cả ba phương pháp đều không dự đoán đúng sản phẩm Cao nào nên F của lớp Cao bằng 0. Xét lớp Thấp, cây đạt F = 0,8 (P = 0,667, R = 1) còn Naïve Bayes đạt F = 0,5. Ở bài này, cây quyết định tốt hơn Naïve Bayes (F trung bình macro 0,4 so với 0,25).

### IV.2 Phân tích cảm xúc

In [6]:
print("Đinh Nhật Huyền Nhân - 6451071055")
evaluate('IV.2 Phân tích cảm xúc')

Đinh Nhật Huyền Nhân - 6451071055
Mâu thuẫn trong dữ liệu: không có

Độ lợi thông tin tại nút gốc:
  giảm: 0.238
  người: 0.401
  chuyển: 0.266
  yêu: 0.266
  vừa: 0.317
  đi: 0.470
Chỉ số Gini tại nút gốc:
  giảm: 0.583
  người: 0.543
  chuyển: 0.574
  yêu: 0.562
  vừa: 0.544
  đi: 0.512

Cây ID3:
[đi = 0..5]
    [người = 11..20]
        => tốt (1 mẫu)
    [người = 6..10]
        [giảm = 0..5]
            => tốt (1 mẫu)
        [giảm = 11..20]
            => xấu (1 mẫu)
    [người = >20]
        => xấu (1 mẫu)
[đi = 11..20]
    [người = 0..5]
        => tốt (1 mẫu)
    [người = 6..10]
        [giảm = 0..5]
            => xấu (1 mẫu)
        [giảm = 11..20]
            => tốt (1 mẫu)
    [người = >20]
        => bình thường (1 mẫu)
[đi = 6..10]
    [người = 0..5]
        => xấu (1 mẫu)
    [người = 11..20]
        => bình thường (1 mẫu)
    [người = 6..10]
        => xấu (1 mẫu)
[đi = >20]
    [chuyển = 11..20]
        => bình thường (2 mẫu)
    [chuyển = 6..10]
        => tốt (1 mẫu)


**Nhận xét:** Bài có ba lớp nên F-measure được tính cho từng lớp rồi lấy trung bình macro. Cây dự đoán đúng 2/4 trạng thái (F trung bình macro 0,267), còn Naïve Bayes đúng cả 4/4 (F = 1).

### Tổng hợp F-measure trung bình macro

In [7]:
print("Đinh Nhật Huyền Nhân - 6451071055")
result = pd.DataFrame(summary)
macro = result[result['Lớp'] == 'Trung bình macro']
print(macro.pivot(index='Bài', columns='Phương pháp', values='F-measure')
      [['ID3', 'CART', 'Naive Bayes']].round(3).to_string())

Đinh Nhật Huyền Nhân - 6451071055
Phương pháp               ID3   CART  Naive Bayes
Bài                                              
III.1 Rủi ro tín dụng   0.400  0.400         1.00
IV.1 Doanh số đồ chơi   0.400  0.400         0.25
IV.2 Phân tích cảm xúc  0.267  0.267         1.00


**Nhận xét:** Naïve Bayes tốt hơn ở bài rủi ro tín dụng và phân tích cảm xúc, còn cây quyết định tốt hơn ở bài doanh số đồ chơi. ID3 và CART tạo ra cùng một cây trên cả ba dữ liệu nên có cùng F-measure. Mỗi tập kiểm thử chỉ có 3-4 mẫu nên kết quả này chưa đủ để kết luận phương pháp nào tốt hơn trong trường hợp tổng quát.